In [1]:
import os
import shutil
from tqdm import tqdm

### ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# passwords
COPY passwords.py .

# functions
COPY functions.py .

# functions_app
COPY functions_app.py .

# preprocessing
COPY preprocessing.py .

# api
COPY api.py .

# copy script into container
COPY app.py .

# parser
COPY cls_parser.pkl .

# copy templates/
COPY ./templates/. /templates/

# copy static
COPY ./static/. /static/

# expose port 5000
EXPOSE 5000

# run script when image is run
CMD ["python", "app.py"]

Writing Dockerfile


### Copy files from local app

In [3]:
list_str_filename = [
    'requirements.txt',
    'passwords.py',
    'functions.py',
    'functions_app.py',
    'preprocessing.py',
    'api.py',
    'app.py',
    'cls_parser.pkl',
]
for str_filename in tqdm(list_str_filename):
    str_origin = f'../02_local_app/{str_filename}'
    str_destination = f'./{str_filename}'
    shutil.copyfile(str_origin, str_destination)

100%|██████████| 8/8 [00:00<00:00, 2206.95it/s]


### Copy directories from local app

In [4]:
list_str_dirname = [
    'templates',
    'static',
]
for str_dirname in tqdm(list_str_dirname):
    str_origin = f'../02_local_app/{str_dirname}'
    str_destination = f'./{str_dirname}'
    try:
        shutil.copytree(str_origin, str_destination)
    except FileExistsError:
        pass

100%|██████████| 2/2 [00:00<00:00, 329.55it/s]


### Build and push to ECR

In [5]:
%%sh

# name the image
image=gen-xii-writeup-app

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  6.672MB
Step 1/16 : FROM python:3.9
 ---> 4b15bb967077
Step 2/16 : RUN apt-get update
 ---> Using cache
 ---> 5c0fe1179326
Step 3/16 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 7bdc9b9d3317
Step 4/16 : COPY requirements.txt .
 ---> Using cache
 ---> 45535078d374
Step 5/16 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 27f3c2e6a8a4
Step 6/16 : COPY passwords.py .
 ---> Using cache
 ---> 621de76b02c1
Step 7/16 : COPY functions.py .
 ---> Using cache
 ---> 000d979e2b22
Step 8/16 : COPY functions_app.py .
 ---> Using cache
 ---> a020e7f99152
Step 9/16 : COPY preprocessing.py .
 ---> Using cache
 ---> 9fc5bcd8864c
Step 10/16 : COPY api.py .
 ---> Using cache
 ---> 87f769663f0f
Step 11/16 : COPY app.py .
 ---> Using cache
 ---> eb23d35f1901
Step 12/16 : COPY cls_parser.pkl .
 ---> Using cache
 ---> e569969964cf
Step 13/16 : COPY ./templates/. /templates/
 ---> Using cache
 ---> 536b2b1f1358
Step 14/16 : COPY ./static/. /stati

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen-xii-writeup-app' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen-xii-writeup-app]
548a561dc022: Preparing
2d17ee5335e6: Preparing
f2dea96f99b0: Preparing
21f8eaefdb89: Preparing
9de27cfd2511: Preparing
1e375f115d6b: Preparing
4ba31654d11b: Preparing
746ae17f92a5: Preparing
31f518b663bd: Preparing
172367f2de5d: Preparing
0f23151a2302: Preparing
85346f64a80d: Preparing
72a23a0307e2: Preparing
afe28ac5c5d1: Preparing
f3b460831925: Preparing
20e2f78dadaf: Preparing
e077e19b6682: Preparing
21e1c4948146: Preparing
68866beb2ed2: Preparing
e6e2ab10dba6: Preparing
0238a1790324: Preparing
1e375f115d6b: Waiting
4ba31654d11b: Waiting
746ae17f92a5: Waiting
31f518b663bd: Waiting
172367f2de5d: Waiting
0f23151a2302: Waiting
85346f64a80d: Waiting
72a23a0307e2: Waiting
afe28ac5c5d1: Waiting
f3b460831925: Waiting
20e2f78dadaf: Waiting
e077e19b6682: Waiting
21e1c4948146: Waiting
68866beb2ed2: Waiting
e6e2ab10dba6: Waiting
0238a1790324: Waiting
2d17ee5335e6: Layer already exists
21f8eaefdb89

### Clean-up

In [6]:
# remove files
list_str_filename.append('Dockerfile')
for str_filename in tqdm(list_str_filename):
    try:
        os.remove(f'./{str_filename}')
    except FileNotFoundError:
        pass

100%|██████████| 9/9 [00:00<00:00, 8797.19it/s]


In [7]:
# remove directories
for str_dirname in tqdm(list_str_dirname):
    try:
        shutil.rmtree(f'./{str_dirname}')
    except FileNotFoundError:
        pass

100%|██████████| 2/2 [00:00<00:00, 816.41it/s]
